# Lumiere Unified SegFormer Retraining

This notebook runs `training/retrain_all.py` in Google Colab for Task218.

Training order: Melasma -> Port Wine Stain -> Vitiligo.

## 1. Upload Training Package

Before opening this notebook in Colab, create `lumiere_retrain_pack.zip` from the repository root:

```bash
zip -r lumiere_retrain_pack.zip \
  training/retrain_all.py \
  training/SegFormer/melasma \
  training/SegFormer/port_wine_stain \
  training/SegFormer/vitiligo \
  training/experiments.json \
  training/experiment_report.md \
  training/checkpoints/SegFormer/segformer_b2_melasma_colab_export \
  data-collection/melasma \
  data-collection/port_wine_stain/processed \
  data-collection/vitiligo \
  requirements.txt \
  -x "*/__pycache__/*" "*.DS_Store"
```

In [ ]:
from google.colab import files

uploaded = files.upload()

## 2. Unzip Package

In [ ]:
!rm -rf /content/Back_Lumiere
!mkdir -p /content/Back_Lumiere
!unzip -q lumiere_retrain_pack.zip -d /content/Back_Lumiere
%cd /content/Back_Lumiere
!find training -maxdepth 3 -type f | sort | head -80

## 3. Install Training Dependencies

In [ ]:
!pip install -q torch torchvision transformers safetensors pillow numpy tqdm

## 4. Check GPU

In [ ]:
import torch

print("cuda_available=", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu=", torch.cuda.get_device_name(0))

## 5. Smoke Test

Run this first to verify paths, datasets, checkpoints, and GPU execution.

In [ ]:
!python training/retrain_all.py \
  --learning-rate 1e-4 \
  --epochs 1 \
  --batch-size 2 \
  --weight-decay 0.01 \
  --run-name smoke_colab \
  --device cuda \
  --num-workers 2

## 6. Full Experiment

After the smoke test succeeds, run the full experiment. Adjust hyperparameters as needed.

In [ ]:
!python training/retrain_all.py \
  --learning-rate 1e-4 \
  --epochs 150 \
  --batch-size 4 \
  --weight-decay 0.01 \
  --run-name experiment_lr1e4 \
  --device cuda \
  --num-workers 2

## 7. Inspect Experiment Log and Report

In [ ]:
import json
from pathlib import Path

experiments = json.loads(Path("training/experiments.json").read_text())
print(json.dumps(experiments[-2:], indent=2))
print("\n--- experiment_report.md ---\n")
print(Path("training/experiment_report.md").read_text()[:4000])

## 8. Package and Download Results

In [ ]:
!zip -r lumiere_retrain_results.zip \
  training/checkpoints/SegFormer/unified \
  training/experiments.json \
  training/experiment_report.md

from google.colab import files
files.download("lumiere_retrain_results.zip")